# Mostrar cartas:

Función que muestra una carta por pantalla, se usa para debug

In [1]:
from IPython.display import display, HTML

def display_card(url):
    # Ahora que tenemos la URL real, generamos el HTML
    html_code = f"""
    <div style="width: 300px; border-radius: 15px; overflow: hidden; box-shadow: 0 8px 16px rgba(0,0,0,0.3);">
        <img src="{url}" alt="Carta" style="width:100%; display: block;">
    </div>
    """
    display(HTML(html_code))

## Crear embeddings:

Funcionalidad para crear embeddings apartir del texto y el nombre de una carta

In [2]:
import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


df = pd.read_csv("cards_final_with_xp.csv")

batch_size=32
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

try:
    texts = [f"{name} \n {text}" for name, text in zip(df["name"], df["text"])]
    query_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size).astype(np.float32)
finally:
    torch.cuda.empty_cache()

c:\venvs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 651.94it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
query_embeddings.shape

(5346, 384)

In [4]:
query_embeddings[:5]


array([[-0.02074328,  0.08789469,  0.06243357, ..., -0.03356438,
        -0.04759217, -0.03279905],
       [ 0.00096978, -0.01984177, -0.04080513, ...,  0.02996143,
        -0.02697211,  0.10606926],
       [-0.02252915,  0.0716615 ,  0.07225666, ..., -0.09477419,
        -0.04195307, -0.01623018],
       [-0.0236518 ,  0.02925191,  0.01507207, ..., -0.06504929,
         0.02110572,  0.04893175],
       [ 0.01233931, -0.03880871, -0.00399657, ...,  0.09457004,
        -0.06981922, -0.02656643]], shape=(5, 384), dtype=float32)

In [5]:
def create_embedding(text):
    return model.encode(text, convert_to_numpy=True).astype(np.float32)

# Embeddings a Redis
Función que recibe un embedding como un array de numpy y devuelve un blob que se le puede pasar a redis

In [6]:
import numpy as np

def to_blob(embedding: np.array) -> bytes:
    """
    Converts embedding to blob.
    :param embedding: embedding
    :return: blob
    """
    return embedding.astype(np.float32).tobytes()

# Iniciar conexión con redis

In [7]:
import redis

# Ejecutar antes docker run --rm --name redis-stack -p 6379:6379 -p 8001:8001 redis/redis-stack:latest
r = redis.Redis(host='localhost', port=6379)


# **Objetivo I**

### **Tarea 1**. Buscar la estructura de datos más apropiada para la cache.

Para esta tarea, la estructura de datos más apropiada para la caché en Redis sería una **tabla Hash**. Esto es ideal para almacenar la información de cada carta, ya que cada carta puede tener múltiples atributos (nombre, texto, tipo, etc.) que pueden ser almacenados como campos dentro del Hash.  Además, los Hashes permiten un acceso rápido a los datos, lo que es crucial para una caché eficiente. Así, tenemos un Hash para cada carta con el nombre de la carta como clave y los atributos como campos dentro del Hash.


### **Tarea 2**. Escribir una función Python que reciba un fichero .csv con un conjunto de ejemplo de cartas y las cargue en una base de datos Redis usando la estructura de datos seleccionada en el paso anterior.


In [8]:
df = pd.read_csv("cards_final_with_xp.csv")

In [9]:
df.head()

,code,name,text,type_code,traits,pack_code,illustrator,image_url,xp,faction_code
0,60401,Jacqueline Fine,[reaction] When an investigator at your locati...,investigator,Clairvoyant,jac,Aleksander Karcz,https://arkhamdb.com/bundles/cards/60401.jpg,0,mystic
1,60402,Arbiter of Fates,Jacqueline Fine deck only. [reaction] When you...,asset,Talent,jac,Pavel Kolomeyets,https://arkhamdb.com/bundles/cards/60402.jpg,0,mystic
2,60403,Dark Future,Revelation - Put Dark Future into play in your...,treachery,Omen|Endtimes,jac,Matt Bradbury,https://arkhamdb.com/bundles/cards/60403.jpg,0,neutral
3,60404,Nihilism,Revelation - Put Nihilism into play in your th...,treachery,Madness,jac,Sara Biddle,https://arkhamdb.com/bundles/cards/60404.jpg,0,neutral
4,60406,Scrying Mirror,Uses (4 secrets). [reaction] After a skill tes...,asset,Item|Charm,jac,Drazenka Kimpel,https://arkhamdb.com/bundles/cards/60406.jpg,0,mystic


In [10]:
def cargar_cartas_en_redis(df: pd.DataFrame) -> None:
    """
    Carga las cartas de un DataFrame en Redis.

    Args:
        df (pd.DataFrame): DataFrame que contiene las cartas.
    """
    for _, serie in df.iterrows():

        data = serie.to_dict()
        code = data.pop("code", None)
        
        if code:
            key = f"card:{code}"
            
        r.hset(key, mapping=data)

In [11]:
cargar_cartas_en_redis(df)

### **Tarea 3**. Escribir funciones python que permita realizar cada una de las acciones. Implementar una por acción.

#### 1. Saber si una carta está en la cache por su campo code.

In [12]:
def carta_en_cache(code: str) -> bool:
    """
    Verifica si una carta está en la cache por su campo code.

    Args:
        code (str): El código de la carta a verificar.

    Returns:
        bool: True si la carta está en la cache, False en caso contrario.
    """
    key = f"card:{code}"
    return r.exists(key) == 1

In [13]:
carta_en_cache("01001")

False

In [14]:
carta_en_cache("06095")

True

#### 2. Recuperar todos los datos de una carta a partir de su code.

In [15]:
def recuperar_carta(code: str) -> bool:
    """
    Recupera todos los datos de una carta a partir de su code.
    
    Args:
        code (str): El código de la carta a recuperar.
        
    Returns:
        dict: Un diccionario con los datos de la carta si está en la cache, None en caso contrario.
    """
    key = f"card:{code}"
    
    if not r.exists(key):
        return None

    data = r.hgetall(key)
    
    return {
        k.decode("utf-8", "ignore"): v.decode("utf-8", "ignore")
        for k, v in data.items()
    }

In [16]:
recuperar_carta("06095")

{'image_url': 'https://arkhamdb.com/bundles/cards/06095.jpg',
 'pack_code': 'tde',
 'type_code': 'treachery',
 'traits': 'Curse',
 'name': 'Deeper Slumber',
 'text': 'Revelation - Put Deeper Slumber into play in your threat area. Your maximum hand size is reduced by 3 and is checked after each time you draw 1 or more cards. [action] [action]: Discard Deeper Slumber.',
 'xp': '0',
 'illustrator': 'Sacha Angel Diener',
 'faction_code': 'mythos'}

#### 3. Meter una carta nueva en la cache.


In [17]:
def meter_carta_en_cache(code: str, data: dict) -> None:
    """
    Mete una carta nueva en la cache si no existe ya.

    Args:
        code (str): El código de la carta a meter.
        data (dict): Un diccionario con los datos de la carta a meter.
    """
    key = f"card:{code}"
    r.hset(key, mapping=data) if r.exists(key) == 0 else print(f"La carta con código {code} ya existe en la cache.")

In [18]:
data_example = {
    "name": "Test Card",
    "text": "This is a test card.",
    "type_code": "test_type",
    "traits": "test_traits",
    "pack_code": "test_pack",
    "faction_code": "test_faction",
    "xp": 0,
    "illustrator": "test_illustrator",
    "image_url": "http://example.com/test_card.jpg"
}

In [19]:
meter_carta_en_cache("01001", data_example)

In [20]:
recuperar_carta("01001")

{'name': 'Test Card',
 'text': 'This is a test card.',
 'type_code': 'test_type',
 'traits': 'test_traits',
 'pack_code': 'test_pack',
 'faction_code': 'test_faction',
 'xp': '0',
 'illustrator': 'test_illustrator',
 'image_url': 'http://example.com/test_card.jpg'}

#### 4. Eliminar una carta de la cache a partir de su campo code.

In [21]:
def eliminar_carta_de_cache(code: str) -> None:
    """
    Elimina una carta de la cache a partir de su campo code.

    Args:
        code (str): El código de la carta a eliminar.
    """
    key = f"card:{code}"
    r.delete(key) if r.exists(key) == 1 else print(f"La carta con código {code} no existe en la cache.")
    

In [22]:
eliminar_carta_de_cache("01001")

In [23]:
recuperar_carta("01001")

# **Objetivo II** 

El equipo del portal ha oído hablar de las capacidades de búsqueda avanzada de Redis y quiere probar si se pueden usar para extender la funcionalidad del portal. En concreto han identificado varias búsquedas que han reclamado los usuarios a lo largo de los años:

- **A**. En el juego hay 7 facciones, 5 que son clases que pueden usar los jugadores (mystic, survivor, guardian, seeker y rogue), la facción “neutral” que es equipo común para todas las clases y la facción “mythos” que son los enemigos. Recientemente se han añadido cartas especiales que tienen más de una facción. Los jugadores están interesados en buscar todas las cartas que contengan varias facciones a la vez, así como, buscar todas las cartas que contengan al menos una de las facciones que indiquen. Por defecto devolvemos las cartas de 5 en 5.
- **B**. Los traits son una forma muy cómoda de buscar cartas interesantes y es muy usada por los
usuarios. No obstante, como hay muchos, manejarlos es algo complicado. Los usuarios
quieren saber cuáles son los traits más comunes para su facción por orden de frecuencia.
Por defecto mostramos páginas con 15 traits.

- **C**. Durante el juego los jugadores ganan puntos de experiencia y pueden gastarlos en mejorar sus mazos. Por eso, es normal que los usuarios quieran buscar cartas que contengan ciertos traits y que puedan usarse para actualizar sus cartas, es decir, que tengan un coste de experiencia (xp) > 0. Normalmente están interesados en cartas que sean de su facción, así que quieren que esas aparezcan entre los primeros resultados, no obstante, también quieren ver cartas de otras facciones ya que hay mazos que mezclan varias facciones. No obstante, nunca quieren ver cartas de la facción “mythos” porque no las pueden incluir en su mazo. Los jugadores también piden filtrar las cartas por lo puntos de experiencia que les quedan, para que no les aparezcan cartas que no pueden comparar. Por defecto devolvemos las cartas de 5 en 5.


### Tarea 1. Diseña un indice que permita realizar las búsquedas anteriores.

Para crear este índice, tenemos que tener en cuenta lo siguiente:

- Como vamos a realizar búsquedas por facción, es importante tener un índice que nos permita buscar por este campo, que será de tipo TAG debido a que los valores son cadenas de texto concretas. Ademas de esto, como las cartas pueden tener varias facciones, es necesario agregar un separador para poder almacenar varias facciones en el mismo campo. En este caso, tendremos que usar el separador “|” para separar las facciones en el campo de facciones. 

- Para la búsqueda por traits, es importante tener un índice que nos permita buscar por este campo, que será de tipo TAG debido a que los valores son cadenas de texto concretas. Ademas de esto, como las cartas pueden tener varios traits, es necesario agregar un separador para poder almacenar varios traits en el mismo campo. En este caso, tendremos que usar el separador “|” para separar los traits en el campo de traits.

- Para la búsqueda por coste de experiencia, es importante tener un índice que nos permita buscar por este campo, que será de tipo NUMERIC debido a que los valores son números enteros. Si además se quiere ordenar resultados por coste de experiencia, conviene marcarlo como **SORTABLE** (aunque en este caso no sería necesario). Con el fin de hacer algunas pruebas, lo pondremos como SORTABLE, realmente no es necesario.  

In [24]:
from redis.exceptions import ResponseError

def crear_indice(nombre_indice: str) -> None:
    """
    Crea un indice en Redis para las cartas.
    """
    try:
        comando = f"""
        FT.CREATE {nombre_indice} 
        ON HASH PREFIX 1 card: 
        SCHEMA faction_code TAG SEPARATOR | 
        traits TAG SEPARATOR | 
        xp NUMERIC SORTABLE
        """
        
        r.execute_command(*comando.split())
        print(f"Índice '{nombre_indice}' creado con éxito.")
        
    except ResponseError as e:
        if "Index already exists" in str(e):
            print(f"El índice '{nombre_indice}' ya existe. No es necesario crearlo de nuevo.")
        else:
            print(f"Error de Redis al crear el índice: {e}")
            
    except Exception as e:
        print(f"Error inesperado: {e}")


In [25]:
crear_indice("cards-idx")

Índice 'cards-idx' creado con éxito.


### Tarea 2. Implementa una función python que realize cada una de las búsquedas anteriores

#### Helpers para parsear respuestas

In [26]:
def _parse_ft_search(raw):
    """
    Parse para FT.SEARCH:
    RESP2: [total, key1, [field, value, ...], key2, [...], ...]
    (y tolera raw=None)
    Devuelve: (total:int, docs:list[dict]) con strings.
    """
    if raw is None:
        return 0, []

    # Si algún día te devolviera dict (RESP3), lo dejamos como fallback
    if isinstance(raw, dict):
        total = int(raw.get("total", 0))
        docs = raw.get("documents", [])
        return total, docs

    def _dec(x):
        return x.decode() if isinstance(x, (bytes, bytearray)) else x

    total = int(raw[0])
    out = []
    i = 1
    while i + 1 < len(raw):
        key = _dec(raw[i])              # "card:1029"
        fields = raw[i + 1]             # [k,v,k,v,...] (bytes normalmente)
        doc = {"_key": key}

        # fields puede venir como list/tuple
        if isinstance(fields, (list, tuple)):
            for j in range(0, len(fields) - 1, 2):
                k = _dec(fields[j])
                v = _dec(fields[j + 1])
                doc[k] = v

        # FIX CLAVE: si no hay 'code' en el HASH/RETURN, sacarlo del docid
        if not doc.get("code"):
            doc["code"] = key.split(":", 1)[1] if ":" in key else key

        out.append(doc)
        i += 2

    return total, out

#### **Objetivo II-A**: búsqueda por facciones

En el enunciado se especifica que quieren: 

- Buscar todas las cartas que contengan varias facciones a la vez (AND): tienen que contener las facciones indicadas, pero pueden contener más. Por ejemplo, si se buscan cartas con las facciones “guardian” y “seeker” en modo AND, se devolverán cartas que tengan ambas facciones, aunque también puedan tener otras facciones adicionales (por ejemplo, una carta con facciones “guardian|seeker|neutral” también sería válida).

- Buscar todas las cartas que contengan al menos una de las facciones que indiquen (OR): tienen que contener al menos una de las facciones indicadas, pero pueden contener más. Por ejemplo, si se buscan cartas con las facciones “guardian” y “seeker” en modo OR, se devolverán cartas que tengan al menos una de esas facciones (por ejemplo, una carta con facciones “guardian|neutral” o “seeker|mythos” también sería válida o incluso alguna con ambas).

In [27]:
def search_by_factions(
    r: redis.Redis,
    factions: list[str],
    mode: str = "OR",            # "OR" o "AND"
    page: int = 0,
    page_size: int = 5,
    index_name: str = "cards-idx",
):
    """
    Devuelve cartas MULTIFACCIÓN (estrictamente más de una facción) que:
    - OR: tengan al menos una de las facciones y al menos otra diferente.
    - AND: tengan todas las facciones (si es solo una, obliga a tener otra).
    Paginado de 5 en 5 (por defecto).
    """
    if not factions:
        raise ValueError("factions no puede estar vacío")

    factions = [f.strip().lower() for f in factions]
    
    # Facciones válidas para multifacción (se excluyen 'neutral' y 'mythos')
    valid_factions = ["mystic", "survivor", "guardian", "seeker", "rogue"]

    if mode.upper() == "AND":
        if len(factions) == 1:
            # Si piden 1 sola, exigimos que tenga esa AND alguna de las demás válidas
            fac = factions[0]
            others = [f for f in valid_factions if f != fac]
            query = f"@faction_code:{{{fac}}}"
        else:
            # Si piden 2 o más con AND, ya es obligatoriamente multifacción por definición
            query = " ".join([f"@faction_code:{{{f}}}" for f in factions])
    else:
        # Modo OR: Para cada facción pedida, construimos la condición de que sea multifacción
        or_blocks = []
        for fac in factions:
            others = [f for f in valid_factions if f != fac]
            # Bloque: (Tiene la facción actual AND tiene al menos una de las otras)
            block = f"(@faction_code:{{{fac}}} @faction_code:{{{'|'.join(others)}}})"
            or_blocks.append(block)
        
        # Unimos todos los bloques con OR lógico a nivel de query
        query = " | ".join(or_blocks)

    offset = page * page_size

    # Ejecutamos el comando manteniendo tu RETURN y LIMIT
    raw = r.execute_command(
        "FT.SEARCH", index_name, query,
        "RETURN", "4", "code", "name", "faction_code", "xp",
        "LIMIT", str(offset), str(page_size)
    )
    
    # Asumo que _parse_ft_search ya la tienes implementada por tu cuenta
    total, docs = _parse_ft_search(raw)
    
    return {
        "total": total, 
        "page": page, 
        "page_size": page_size, 
        "results": docs, 
        "query": query
    }

In [28]:
search_by_factions(r, ["seeker"], mode="AND", page=0, page_size=5)

{'total': 31,
 'page': 0,
 'page_size': 5,
 'results': [{'_key': 'card:4022',
   'name': 'Ancient Stone',
   'faction_code': 'seeker',
   'xp': '1',
   'code': '4022'},
  {'_key': 'card:2021',
   'name': 'Strange Solution',
   'faction_code': 'seeker',
   'xp': '0',
   'code': '2021'},
  {'_key': 'card:9040',
   'name': 'Alchemical Distillation',
   'faction_code': 'seeker',
   'xp': '0',
   'code': '9040'},
  {'_key': 'card:10056',
   'name': 'Prismatic Spectacles',
   'faction_code': 'seeker',
   'xp': '2',
   'code': '10056'},
  {'_key': 'card:10050',
   'name': 'Transmogrify',
   'faction_code': 'seeker',
   'xp': '0',
   'code': '10050'}],
 'query': '@faction_code:{seeker}'}

### Objetivo II-B: traits más comunes por facción

La consulta es bastante sencilla, solamente hay que tener en cuenta que hay que separar cada uno de los traits que aparecen en las cartas, ya que cada carta puede tener varios traits separados por el caracter “|”. Para esto, se puede usar la función SPLIT para separar los traits y luego contar la frecuencia de cada uno de ellos. En caso de que no se hiciera, entonces contaría el conjunto de traits como un único trait, lo cual no es lo que queremos.

In [29]:
def top_traits_for_faction(
    r,
    faction: str,
    page: int = 0,
    page_size: int = 15,
    index_name: str = "cards-idx",
):
    """
    Devuelve los traits más comunes entre las cartas de una facción dada, ordenados por frecuencia.
    """
    faction = faction.strip().lower()
    offset = page * page_size

    query = f"@faction_code:{{{faction}}}"

    try:
        raw = r.execute_command(
            "FT.AGGREGATE", index_name, query,
            "LOAD", "1", "traits",
            "APPLY", "split(@traits, '|')", "AS", "trait",
            "GROUPBY", "1", "@trait",
            "REDUCE", "COUNT", "0", "AS", "frecuencia",
            "SORTBY", "2", "@frecuencia", "DESC",
            "LIMIT", str(offset), str(page_size)
        )

        total_groups = raw[0]
        results = []

        for i in range(1, len(raw)):
            row_data = raw[i]
            
            res_dict = {}
            for j in range(0, len(row_data), 2):
                key = row_data[j].decode('utf-8') if isinstance(row_data[j], bytes) else row_data[j]
                val = row_data[j+1].decode('utf-8') if isinstance(row_data[j+1], bytes) else row_data[j+1]
                res_dict[key] = val
            
            t_name = res_dict.get("trait")
            
            if t_name and t_name.strip():
                results.append((t_name.strip(), int(res_dict.get("frecuencia", 0))))

        return {
            "faction": faction,
            "total_traits": total_groups,
            "page": page,
            "page_size": page_size,
            "results": results,
        }

    except Exception as e:
        print(f"Error ejecutando FT.AGGREGATE: {e}")
        return None

In [30]:
top_traits_for_faction(r, "survivor", page=0, page_size=15)

{'faction': 'survivor',
 'total_traits': 39,
 'page': 0,
 'page_size': 15,
 'results': [('Item', 15),
  ('Fortune', 7),
  ('Tool', 7),
  ('Spirit', 7),
  ('Blessed', 6),
  ('Tactic', 6),
  ('Innate', 6),
  ('Weapon', 6),
  ('Melee', 5),
  ('Developed', 4),
  ('Ally', 4),
  ('Charm', 4),
  ('Trick', 4),
  ('Dilemma', 3),
  ('Cursed', 3)]}

### Objetivo II-C: buscar cartas con xp>0, excluir mythos, filtrar por xp máximo, priorizar facción y permitir mezcla


Una primera idea para realizar esta consulta sería hacer una consulta normal con FT.SEARCH y luego ordenar los resultados en python para priorizar las cartas de la facción preferida. No obstante, esto no es muy eficiente, ya que tendríamos que recuperar muchos resultados para luego ordenarlos en python.

Otra idea sería realizar dos consultas, una para la facción preferida y otra para el resto de facciones, y luego mezclar los resultados en python. Esto sería un poco más eficiente, pero todavía tendríamos que realizar dos consultas.

Por último, la mejor idea sería usar FT.AGGREGATE para realizar la consulta y usar la función CASE (que evalúa una condición y devuelve un valor) para asignar una puntuación a cada carta en función de si pertenece a la facción preferida o no, y luego ordenar los resultados por esa puntuación. De esta forma, las cartas de la facción preferida aparecerían primero, pero también se incluirían cartas de otras facciones que cumplan los requisitos. Además, al usar FT.AGGREGATE, solo recuperaríamos los resultados que necesitamos, lo que sería más eficiente.

Usando la única consulta con FT.AGGREGATE (para más información del uso de la función CASE, consultar la documentación de Redisearch: https://redis.io/docs/latest/develop/ai/search-and-query/advanced-concepts/aggregations/#case, en el subapartado de APPLY):  

In [31]:
def search_upgrades(
    r,
    traits=None,
    preferred_faction=None,
    xp_max=None,
    page=0,
    page_size=5,
    index_name="cards-idx",
):  
    
    # Función auxiliar para convertir bytes a string (si es necesario)
    def _to_str(v):
        return v.decode("utf-8", errors="ignore") if isinstance(v, (bytes, bytearray)) else str(v)

    # Función auxiliar para parsear resultados de FT.AGGREGATE 
    def _parse_ft_aggregate(raw):
        if not raw:
            return 0, []
        total = int(_to_str(raw[0]))
        docs = []
        for row in raw[1:]:
            if not isinstance(row, (list, tuple)):
                continue
            d = {}
            for i in range(0, len(row), 2):
                if i + 1 >= len(row):
                    break
                k = _to_str(row[i])
                v = row[i + 1]
                d[k] = _to_str(v)
            if "xp" in d:
                try:
                    d["xp"] = int(d["xp"])
                except Exception:
                    pass
            docs.append(d)
        return total, docs

    # Validar y normalizar parámetros de paginación
    page = max(0, int(page))
    page_size = max(1, int(page_size))
    offset = page * page_size

    # construir filtros base 
    if xp_max is None:
        xp_filter = "@xp:[(1 +inf]"
    else:
        xp_filter = f"@xp:[(1 {int(xp_max)}]"

    parts = [xp_filter, "-@faction_code:{mythos}"]

    # construir filtro de traits
    if traits:
        if isinstance(traits, str):
            traits = [traits]
        traits_clean = [t.strip() for t in traits if t and t.strip()]
        if traits_clean:
            parts.append(f"@traits:{{{'|'.join(traits_clean)}}}")

    # construir consulta base (sin preferida)    
    base_query = " ".join(filter(None, parts)) if parts else "*"

    # manejar facción preferida (si es mythos o None, no se prioriza ninguna)
    preferred = preferred_faction.strip() if preferred_faction else None
    if preferred and preferred.lower() == "mythos":
        preferred = None
    
    # No hay preferida
    if not preferred:
        raw = r.execute_command(
            "FT.SEARCH",
            index_name,
            base_query,
            "RETURN", "4", "code", "name", "faction_code", "xp",
            "SORTBY", "xp", "ASC",
            "LIMIT", str(offset), str(page_size),
        )
        total, docs = _parse_ft_search(raw)
        return {
            "faction": preferred_faction,
            "total_cards": total,
            "page": page,
            "page_size": page_size,
            "results": docs,
            "method": "single_query",
        }

    # con preferida 
    pref_escaped = preferred.replace("\\", "\\\\").replace("'", "\\'")
    case_expr = f"case(@faction_code == '{pref_escaped}', 1, 0)"
    r.execute_command("FT.CONFIG SET ENABLE_UNSTABLE_FEATURES true") # necesario para usar case en aggregate
    raw = r.execute_command(
        "FT.AGGREGATE",
        index_name,
        base_query,
        "LOAD", "4", "@code", "@name", "@faction_code", "@xp",
        "APPLY", case_expr, "AS", "has_pref",
        "APPLY", "1 - @has_pref", "AS", "priority",
        "SORTBY", "4", "@priority", "ASC", "@xp", "DESC",
        "LIMIT", str(offset), str(page_size),
    )

    # En caso de que no se quiera priorizar solo las cartas exclusivamente de la facción preferida, 
    # sino también las que la incluyen entre varias, se podría usar esta consulta alternativa: 
    """
    pref_escaped = preferred.replace("\\", "\\\\").replace("'", "\\'")
    raw = r.execute_command(
        "FT.AGGREGATE",
        index_name,
        base_query,
        "LOAD", "4", "@code", "@name", "@faction_code", "@xp",
        "APPLY", f"contains(@faction_code, '{pref_escaped}') > 0", "AS", "has_pref",
        "APPLY", "1 - @has_pref", "AS", "priority",
        "SORTBY", "4", "@priority", "ASC", "@xp", "DESC",
        "LIMIT", str(offset), str(page_size),
    )
    """

    total, docs = _parse_ft_aggregate(raw)

    # eliminar auxiliares si aparecen
    for d in docs:
        d.pop("has_pref", None)
        d.pop("priority", None)

    return {
        "faction": preferred_faction,
        "total_cards": total,
        "page": page,
        "page_size": page_size,
        "results": docs,
    }

In [32]:
search_upgrades(r, traits=["weapon"], preferred_faction="guardian", xp_max=5, page=0, page_size=5)

{'faction': 'guardian',
 'total_cards': 194,
 'page': 0,
 'page_size': 5,
 'results': [{'name': 'Miracle Wish', 'faction_code': 'guardian', 'xp': 5},
  {'name': 'Butterfly Swords', 'faction_code': 'guardian', 'xp': 5},
  {'name': 'One-Two Punch', 'faction_code': 'guardian', 'xp': 5},
  {'name': 'Monster Slayer', 'faction_code': 'guardian', 'xp': 5},
  {'name': 'Armor of Ardennes', 'faction_code': 'guardian', 'xp': 5}]}

Utilizando las dos consultas con FT.SEARCH:

In [33]:
def search_upgrades(
    r,
    traits=None,
    preferred_faction=None,
    xp_max=None,
    page=0,
    page_size=5,
    index_name="cards-idx",
):
    try:
        traits_query = '|'.join(traits)

        query_main = f"@traits:{{{traits_query}}} @xp:[(0 {xp_max}] @faction_code:{{{preferred_faction}}} -@faction_code:{{mythos}}"

        res = r.execute_command(
            "FT.SEARCH",
            index_name,
            query_main,
            "SORTBY", "xp", "DESC",
            "LIMIT", page * page_size, page_size,
            "RETURN", 4, "name", "faction_code", "traits", "xp"
        )

        total = res[0]
        results = []

        def parse_results(raw):
            docs = []
            for i in range(1, len(raw), 2):
                key = raw[i].decode("utf-8")
                fields = raw[i + 1]

                doc = {"_key": key}
                for j in range(0, len(fields), 2):
                    field = fields[j].decode("utf-8")
                    value = fields[j + 1].decode("utf-8")
                    doc[field] = value

                # extraemos el code del _key
                doc["code"] = key.split(":")[-1]
                docs.append(doc)
            return docs

        # Parseamos primera búsqueda
        results.extend(parse_results(res))

        # Si no hay suficientes resultados, completamos con otras facciones
        if (page + 1) * page_size > total:
            resto = total % page_size

            query_secondary = f"@traits:{{{traits_query}}} @xp:[(0 {xp_max}] -@faction_code:{{mythos}} -@faction_code:{{{preferred_faction}}}"

            if page * page_size < total:
                offset = 0
                limit = resto
            else:
                offset = resto + page_size * (abs((page + 1) * page_size - total) // page_size)
                limit = page_size

            res2 = r.execute_command(
                "FT.SEARCH",
                index_name,
                query_secondary,
                "SORTBY", "xp", "DESC",
                "LIMIT", offset, limit,
                "RETURN", 4, "name", "faction_code", "traits", "xp"
            )

            results.extend(parse_results(res2))

        # Nos aseguramos de no devolver más de page_size
        results = results[:page_size]

        return {
            "total": total,
            "page": page,
            "page_size": page_size,
            "results": results,
            "query": query_main
        }

    except Exception as e:
        print(f'Error al realizar la búsqueda: {e}')
        return None

In [34]:
search_upgrades(
    r,
    traits=["weapon"],
    preferred_faction="guardian",
    xp_max=5,
    page=4,
    page_size=5
)

{'total': 12,
 'page': 4,
 'page_size': 5,
 'results': [{'_key': 'card:10077',
   'xp': '2',
   'name': 'British Bull Dog',
   'faction_code': 'rogue',
   'traits': 'Item|Weapon|Firearm|Illicit',
   'code': '10077'},
  {'_key': 'card:08118',
   'xp': '2',
   'name': 'Enchanted Bow',
   'faction_code': 'mystic|survivor',
   'traits': 'Spell|Blessed|Weapon|Ranged',
   'code': '08118'},
  {'_key': 'card:10117',
   'xp': '1',
   'name': 'Hatchet',
   'faction_code': 'survivor',
   'traits': 'Item|Tool|Weapon|Ranged',
   'code': '10117'}],
 'query': '@traits:{weapon} @xp:[(0 5] @faction_code:{guardian} -@faction_code:{mythos}'}

Hay algunas diferencias (como por ejemplo, que consideramos que los que solamente son de la facción preferida tienen mayor prioridad que los que son de varias facciones incluyendo la preferida, aunque esto se podría cambiar fácilmente, por ejemplo, usando el operador contains(facción preferida) > 0, y ordenándolos), pero en general el resultado es bastante similar. No es necesario ordenarlos por la experiencia, pero en este caso se ha realizado para comparar más o menos los resultados que se obtienen con ambas consultas. Usaremos la última función para realizar las pruebas.

### Tarea 3. ¿Cómo comprobarias que has implementado correctamente las búsquedas anteriores?

Al aplicar las funciones de búsqueda, vemos que los resultados que se devuelven cumplen con los criterios de búsqueda especificados. Por ejemplo, al buscar cartas con las facciones “guardian” y “seeker” en modo AND, verificamos que todas las cartas devueltas contienen ambas facciones, aunque también puedan tener otras facciones adicionales. De manera similar, al buscar cartas con las facciones “guardian” y “seeker” en modo OR, verificamos que todas las cartas devueltas contienen al menos una de esas facciones.

Por si acaso, vamos a usar varias funciones para comprobar que los resultados son correctos (realizaremos algunos tests controlados para verificar el comportamiento de las funciones). 

In [35]:
def _recrear_indice_pruebas(index_name="cards-idx"):
    try:
        r.execute_command("FT.DROPINDEX", index_name)
    except Exception:
        pass

    r.execute_command(
        "FT.CREATE", index_name,
        "ON", "HASH",
        "PREFIX", "1", "card:",
        "SCHEMA",
        "code", "TAG",
        "name", "TEXT",
        "faction_code", "TAG", "SEPARATOR", "|",
        "traits", "TAG", "SEPARATOR", "|",
        "xp", "NUMERIC", "SORTABLE"
    )

def cargar_datos_inventados():
    r.flushdb()
    _recrear_indice_pruebas("cards-idx")
    print("Cargando datos inventados...")

    cartas_mock = [
        {"code": "TEST01", "name": "Hechizo Trucado", "faction_code": "mystic|rogue", "traits": "Spell|Trick", "xp": "1"},
        {"code": "TEST02", "name": "Espada Sagrada", "faction_code": "guardian", "traits": "Item|Weapon|Melee", "xp": "3"},
        {"code": "TEST03", "name": "Escopeta Vieja", "faction_code": "survivor", "traits": "Item|Weapon|Firearm", "xp": "2"},
        {"code": "TEST04", "name": "Lupa", "faction_code": "seeker", "traits": "Item|Tool", "xp": "0"},
        {"code": "TEST05", "name": "Monstruo Terrible", "faction_code": "mythos", "traits": "Monster|Elite", "xp": "0"},
        {"code": "TEST06", "name": "Amuleto Raro", "faction_code": "neutral", "traits": "Relic", "xp": "1"},
    ]

    for carta in cartas_mock:
        r.hset(f"card:{carta['code']}", mapping=carta)

    print(f"Se han cargado {len(cartas_mock)} cartas de prueba.\n")

def probar_busqueda_a_facciones():
    print("--- PRUEBA A: Búsqueda por Facciones ---")
    print("Resultado previsto: Hechizo Trucado")
    resultados = search_by_factions(r, factions=["rogue", "mystic"], mode="AND")
    print(resultados, "\n")

def probar_busqueda_b_traits():
    print("--- PRUEBA B: Traits más comunes ---")
    print("Resultado previsto: Weapon, Item, Melee, con frecuencia 1 cada uno (en cualquier orden)")
    resultados = top_traits_for_faction(r, "guardian", page=0, page_size=15)
    print(resultados, "\n")

def probar_busqueda_c_mejoras():
    print("--- PRUEBA C: Mejora de Mazo ---")
    print("Buscamos: faccion_preferida='survivor', traits=['Weapon'], xp_max=2")
    print("Esperado: Debería devolver 'Escopeta Vieja' (TEST03).")
    print("NO debe devolver 'Espada Sagrada' (cuesta 3 XP, supera el límite).")
    print("NO debe devolver 'Lupa' (cuesta 0 XP).")
    print("NO debe devolver 'Monstruo Terrible' (es facción mythos).")
    resultados = search_upgrades(
        r,
        traits=["weapon"],
        preferred_faction="survivor",
        xp_max=2,
        page=0,
        page_size=5
    )
    print(resultados, "\n")

cargar_datos_inventados()
probar_busqueda_a_facciones()
probar_busqueda_b_traits()
probar_busqueda_c_mejoras()


Cargando datos inventados...
Se han cargado 6 cartas de prueba.

--- PRUEBA A: Búsqueda por Facciones ---
Resultado previsto: Hechizo Trucado
{'total': 1, 'page': 0, 'page_size': 5, 'results': [{'_key': 'card:TEST01', 'code': 'TEST01', 'name': 'Hechizo Trucado', 'faction_code': 'mystic|rogue', 'xp': '1'}], 'query': '@faction_code:{rogue} @faction_code:{mystic}'} 

--- PRUEBA B: Traits más comunes ---
Resultado previsto: Weapon, Item, Melee, con frecuencia 1 cada uno (en cualquier orden)
{'faction': 'guardian', 'total_traits': 3, 'page': 0, 'page_size': 15, 'results': [('Melee', 1), ('Weapon', 1), ('Item', 1)]} 

--- PRUEBA C: Mejora de Mazo ---
Buscamos: faccion_preferida='survivor', traits=['Weapon'], xp_max=2
Esperado: Debería devolver 'Escopeta Vieja' (TEST03).
NO debe devolver 'Espada Sagrada' (cuesta 3 XP, supera el límite).
NO debe devolver 'Lupa' (cuesta 0 XP).
NO debe devolver 'Monstruo Terrible' (es facción mythos).
{'total': 1, 'page': 0, 'page_size': 5, 'results': [{'_key': 

Vemos que los resultados que nos dan son los que queremos. 

In [36]:
r = redis.Redis(host='localhost', port=6379)
r.flushdb()
cargar_cartas_en_redis(df)
crear_indice("cards-idx")

Índice 'cards-idx' creado con éxito.


# **Objetivo III**

El equipo del portal quiere implementar un recomendador automático de cartas. La idea es que, dada una carta, el sistema muestre otras 5 cartas parecidas a dicha carta que a los usuarios les pudieran interesar. Para ello, se ha consultado con el equipo de diseño de “Arkham Dread” y han dicho que lo mejor sería utilizar el nombre y el texto de la carta, si no fuera posible también indican que los traits son una alternativa, aunque son menos específicos generales. Al preguntarles por las ilustraciones, el equipo de diseño ha respondido que están más pensadas como decoración y no dan información sobre lo que hace la carta. Con esta información el equipo de desarrollo ha planteado 4 alternativas de más sencilla a más compleja:
- **A.** Usar solo metadatos de la carta y obviar el nombre y el texto: Proponen buscar cartas de
la misma facción y que tengan alguno de los traits de la carta sobre la que se realiza la
búsqueda.
- **B.** Capacidades full-text de redis: Aprovechar las capacidades de redis y hace una busqueda
full-text con el texto de la carta.
- **C.** Búsqueda semántica: Hacer embeddings con el nombre y el texto de la carta para luego
hacer búsqueda semántica.


### Tarea 1. Diseña un índice que permita realizar las alternativas anteriores.

Como tenemos menos de 1 millón de datos, en lugar de HNSW podemos usar un índice de tipo FLAT, que es el índice por defecto en Redisearch. Este tipo de índice es adecuado para conjuntos de datos pequeños y medianos, y nos permitirá realizar búsquedas semánticas utilizando los embeddings generados a partir del nombre y el texto de la carta. Sin embargo, asumimos (como este es un ejercicio académico) que la aplicación se usará para más de 1 millón de cartas, por lo que en un entorno real sería recomendable usar un índice de tipo HNSW para mejorar el rendimiento de las búsquedas semánticas.

In [37]:
import redis

r = redis.Redis(host="localhost", port=6379, decode_responses=False)

def crear_indice_objetivo3(index_name="idx_cards", dim=384):
    """
    Índice único para:
      A) TAG: faction_code, traits (multi-valor con '|')
      B) TEXT: name, text
      C) VECTOR: embedding (KNN)
    """
    try:
        r.execute_command(
            "FT.CREATE", index_name,
            "ON", "HASH",
            "PREFIX", "1", "card:",
            "SCHEMA",
            "faction_code", "TAG", "SEPARATOR", "|",
            "traits", "TAG", "SEPARATOR", "|",
            "name", "TEXT",
            "text", "TEXT",
            "embedding", "VECTOR", "HNSW", "6",
                "TYPE", "FLOAT32",
                "DIM", str(dim),
                "DISTANCE_METRIC", "COSINE"
        )
        print("Índice creado:", index_name)
    except Exception as e:
        print("No se pudo crear:", e)


In [38]:
crear_indice_objetivo3()

Índice creado: idx_cards


### Tarea 2. Haz una función python que implemente cada una de las alternativas propuestas. La función debe recibir el código de una carta e imprimir por pantalla 5 otras cartas parecidas.

In [39]:
import re
import numpy as np

def get_card_no_embedding(code: str) -> dict:
    """Lee card:<code> y devuelve dict (str->str), ignorando embedding (binario)."""
    d = r.hgetall(f"card:{code}")
    if not d:
        raise ValueError(f"No existe card:{code} en Redis")

    out = {}
    for k, v in d.items():
        ks = k.decode()

        # No intentamos decodificar embedding (binario), lo dejamos como bytes para no romperlo. 
        if ks == "embedding":
            continue

        out[ks] = v.decode(errors="ignore")  # por si hay algún carácter raro en text
    return out

def imprimir_docs(titulo, docs):

    print(titulo)

    for i, doc in enumerate(docs, 1):
        extra = f" | score={float(doc['score']):.4f}" if "score" in doc else ""
        print(f"{i:02d}) [{doc.get('code')}] {doc.get('name')}{extra}")
        print(f"    Facción: {doc.get('faction_code')} | Traits: {doc.get('traits')} | Name: {doc.get('name')} | Text: {doc.get('text')}")
        display_card(doc.get("image_url", ""))

### A) Misma facción y comparte alguno de los traits

In [40]:
def recomendar_A(code: str, k: int = 5, index_name="idx_cards"):

    # Obtenermos los datos de la carta. 
    base = get_card_no_embedding(code)
    faction_raw = (base.get("faction_code") or "").strip()
    traits = [t for t in (base.get("traits") or "").split("|") if t]

    # En caso de que no tenga ni facción de traits, no podemos recomendar por metadatos. 
    if not faction_raw or not traits:
        print("No hay faction o traits en la carta base.")
        return

    # Si la carta base fuese multi-facción, buscaríamos cartas que compartan al menos una de esas facciones 
    # y al menos uno de esos traits.
    factions = [f for f in faction_raw.split("|") if f]
    faction_or = "|".join(factions)

    traits_or = "|".join(traits)

    query = f"@faction_code:{{{faction_or}}} @traits:{{{traits_or}}} -@code:{{{code}}}"

    # Pedimos k
    res = r.execute_command(
        "FT.SEARCH", index_name, query,
        "RETURN", "5", "name", "faction_code", "traits", "type_code", "image_url",
        "LIMIT", "0", str(k)
    )


    # Parseamos resultado (total y lista de docs)
    total, docs = _parse_ft_search(res)

    # Filtrar carta base
    docs = [d for d in docs if str(d.get("code")) != str(code)]
    docs = docs[:k]

    print(f"Total candidatos: {total - 1}")
    imprimir_docs("Recomendación por metadatos", docs)

### B) Full-text con el texto de la carta

In [41]:
#!pip install nltk

In [42]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')


def recomendar_B(code: str, k: int = 5, index_name="idx_cards"):
    base = get_card_no_embedding(code)
    text = (base.get("text") or "")
    name = (base.get("name") or "")

    stop_words = set(stopwords.words('english'))
    
    s = (name + " " + text).lower()
    s = re.sub(r"[^a-z0-9áéíóúüñ\s]", " ", s)
    words = [w for w in s.split() if w not in stop_words]

    if not words:
        print("B) No hay texto útil.")
        return

    or_part = "|".join(words)
    query = f"(@text:({or_part}) ) -@code:{{{code}}}" # Sale mejor si no incluimos el name en la búsqueda, porque suele ser muy específico y no aporta tanto a la similitud semántica.
    #     query = f"(@name:({or_part}) | @text:({or_part}) ) -@code:{{{code}}}"
    res = r.execute_command(
        "FT.SEARCH", index_name, query,
        "RETURN", "3", "name", "text", "image_url",
        "LIMIT", "0", str(k + 1)
    )

    total, docs = _parse_ft_search(res)

    # Filtrar carta base
    docs = [d for d in docs if str(d.get("code")) != str(code)]
    docs = docs[:k]

    print(f"Total candidatos: {total - 1}")
    imprimir_docs("Recomendación full-text", docs)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sheng\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### C) Búsqueda semántica (embeddings + KNN)

In [43]:
from redis.commands.search.query import Query
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

def asegurar_embeddings(pattern="card:*", batch=200):
    """Genera embeddings solo si faltan. Se ejecuta una vez."""
    pipe = r.pipeline()
    n = 0
    for key in r.scan_iter(pattern):
        if r.hexists(key, "embedding"):
            continue
        h = r.hgetall(key)
        name = h.get(b"name", b"").decode(errors="ignore")
        text = h.get(b"text", b"").decode(errors="ignore")
        s = (name + " " + text).strip()
        if not s:
            continue
        vec_bytes = model.encode([s], normalize_embeddings=True)[0].astype(np.float32).tobytes()
        pipe.hset(key, mapping={"embedding": vec_bytes})
        n += 1
        if n % batch == 0:
            pipe.execute()
    pipe.execute()
    if n > 0:
        print(f"Embeddings guardados en {n} cartas.")
    else:
        print("Todos los embeddings ya estaban en Redis.")

def recomendar_C(code: str, k: int = 5, index_name="idx_cards"):
    asegurar_embeddings()

    base = get_card_no_embedding(code)
    text = f"{base.get('name', '')} {base.get('text', '')}".strip()

    if not text:
        print("La carta base no tiene texto suficiente.")
        return

    vec = model.encode([text], normalize_embeddings=True)[0].astype(np.float32)

    q = (
        Query(f"*=>[KNN {k + 1} @embedding $vec AS score]")
        .sort_by("score")
        .return_fields("code", "name", "score", "faction_code", "traits", "text", "image_url")
        .dialect(2)
    )
    res = r.ft(index_name).search(q, query_params={"vec": vec.tobytes()})

    docs = []
    for doc in res.docs:
        code_found = getattr(doc, "code", None) or doc.id.split(":", 1)[-1]
        if str(code_found) == str(code):
            continue
        docs.append({
            "code": code_found,
            "name": getattr(doc, "name", None),
            "score": float(getattr(doc, "score", 0.0)),
            "faction_code": getattr(doc, "faction_code", None),
            "traits": getattr(doc, "traits", None),
            "text": getattr(doc, "text", None),
            "image_url": getattr(doc, "image_url", None),
        })
        if len(docs) >= k:
            break
    
    print(f"Devueltos: {len(docs)} (KNN)")
    imprimir_docs("Recomendación semántica (KNN)", docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 662.17it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Tarea 3. Comprueba los resultados obtenidos ¿Qué diferencias hay entre las distintas alternativas? ¿Cuántas cartas es capaz de devolver cada aproximación? ¿Qué tal es el orden que devuelve? Da tu opinión justificada en los resultados obtenidos. Haz las pruebas usando la carta con código 1029 (shotgun).

In [44]:
code = "1029"

In [45]:
recomendar_A(code)

Total candidatos: 65
Recomendación por metadatos
01) [5115] .45 Thompson
    Facción: guardian|rogue | Traits: Item|Weapon|Firearm|Illicit | Name: .45 Thompson | Text: None


02) [2226] Springfield M1903
    Facción: guardian | Traits: Item|Weapon|Firearm | Name: Springfield M1903 | Text: None


03) [2301] Lightning Gun
    Facción: guardian | Traits: Item|Weapon|Firearm | Name: Lightning Gun | Text: None


04) [7108] Riot Whistle
    Facción: guardian | Traits: Item|Tool | Name: Riot Whistle | Text: None


05) [2147] Bandolier
    Facción: guardian | Traits: Item | Name: Bandolier | Text: None


In [46]:
recomendar_B(code) 

Total candidatos: 4847
Recomendación full-text
01) [08088] Old Shotgun
    Facción: None | Traits: None | Name: Old Shotgun | Text: Uses (0 ammo). While playing an event, treat Old Shotgun's uses value as 2. [action] Spend 1 ammo: Fight. You get +3 [combat] for this attack. Instead of its standard damage, this attack's damage is equal to the amount you succeed by, or fail by if you fail and would damage another investigator (to a minimum of 1, to a maximum of 3).


02) [08732] Cookie's Custom .32
    Facción: None | Traits: None | Name: Cookie's Custom .32 | Text: Fast. Uses (2 ammo). [action] Spend 1 ammo: Fight. Either fight with a base [combat] skill of 5, or get +2 [combat] for this attack. This attack deals +1 damage.


03) [53006] Colt Vest Pocket
    Facción: None | Traits: None | Name: Colt Vest Pocket | Text: Uses (5 ammo). [action] Spend 1 ammo: Fight. You get +2 [combat] for this attack. This attack deals +1 damage. Forced - At the end of the round, if you triggered Colt Vest Pocket's "[action]" ability: Discard it.


04) [1047] .41 Derringer
    Facción: None | Traits: None | Name: .41 Derringer | Text: Uses (3 ammo). [action] Spend 1 ammo: Fight. You get +2 [combat] for this attack. If you succeed by 2 or more, this attack deals +1 damage.


05) [6327] Sawed-Off Shotgun
    Facción: None | Traits: None | Name: Sawed-Off Shotgun | Text: Uses (2 ammo). [action] Spend 1 ammo: Fight. Instead of its standard damage, this attack deals 1 damage for each point you succeed by (to a minimum of 1, to a maximum of 6). If you fail and would damage another investigator, this attack deals 1 damage for each point you fail by (to a minimum of 1, to a maximum of 6).


In [47]:
recomendar_C(code)

Embeddings guardados en 5324 cartas.
Devueltos: 5 (KNN)
Recomendación semántica (KNN)
01) [08088] Old Shotgun | score=0.1506
    Facción: guardian|rogue | Traits: Item|Weapon|Firearm | Name: Old Shotgun | Text: Uses (0 ammo). While playing an event, treat Old Shotgun's uses value as 2. [action] Spend 1 ammo: Fight. You get +3 [combat] for this attack. Instead of its standard damage, this attack's damage is equal to the amount you succeed by, or fail by if you fail and would damage another investigator (to a minimum of 1, to a maximum of 3).


02) [6327] Sawed-Off Shotgun | score=0.2253
    Facción: rogue | Traits: Item|Weapon|Firearm|Illicit | Name: Sawed-Off Shotgun | Text: Uses (2 ammo). [action] Spend 1 ammo: Fight. Instead of its standard damage, this attack deals 1 damage for each point you succeed by (to a minimum of 1, to a maximum of 6). If you fail and would damage another investigator, this attack deals 1 damage for each point you fail by (to a minimum of 1, to a maximum of 6).


03) [11032] Remington Model 1858 | score=0.3365
    Facción: guardian | Traits: Item|Weapon|Firearm | Name: Remington Model 1858 | Text: Uses (3 ammo). [action] Spend 1 ammo: Fight. You get +2 [combat] for this attack. This attack deals +1 damage. [reaction] When Remington Model 1858 enters or leaves play: Immediately trigger the above [action] ability, ignoring all costs.


04) [11022] Remington Model 1858 | score=0.3394
    Facción: guardian | Traits: Item|Weapon|Firearm | Name: Remington Model 1858 | Text: Uses (2 ammo). [action] Spend 1 ammo: Fight. You get +1 [combat] for this attack. This attack deals +1 damage. [reaction] When Remington Model 1858 leaves play: Immediately trigger the above [action] ability, ignoring all costs.


05) [4273] Old Hunting Rifle | score=0.3866
    Facción: survivor | Traits: Item|Weapon|Firearm | Name: Old Hunting Rifle | Text: Uses (3 ammo). [action] Spend 1 ammo: Fight. You get +3 [combat] and deal +2 damage for this attack. If a [skull] or [auto_fail] symbol is revealed during this attack, the rifle jams. (This attack automatically fails. Before you can activate this ability again, you must perform the following ability: "[action]: You clear the jam.")


**1) ¿Qué diferencias hay entre las distintas alternativas?**

A (metadatos) filtra por facción y traits compartidos con la carta base (Shotgun es guardian, con traits Item, Weapon y Firearm). Solo devuelve cartas que cumplan ambas condiciones, por lo que los resultados son todos armas de fuego del mismo perfil. 

B (full-text) busca por las palabras del texto de la carta (ammo, spend, fight, combat, attack, damage...). Al ser términos muy comunes en cartas de combate, devuelve un conjunto enorme de candidatos (más de 4000) que incluye no solo armas de fuego sino cualquier carta que mencione esas palabras. Esto explica que aparezcan cartas que comparten vocabulario de combate pero no son necesariamente "parecidas" a una escopeta.

C (semántica) compara por significado usando embeddings. Los resultados son los más coherentes: Old Shotgun, Sawed-Off Shotgun, dos versiones de Remington Model 1858 y Old Hunting Rifle. Todas son armas de fuego con mecánicas muy similares a Shotgun (gastar munición, atacar, daño variable).

**2) ¿Cuántas cartas es capaz de devolver cada aproximación?**

Con la carta 1029 (Shotgun):

A: encuentra 65 candidatos. Es la más restrictiva porque exige coincidencia exacta en facción Y en cada uno de los traits.

B: encuentra más de 4000 candidatos. Es la más amplia porque basta con que el texto de la carta contenga alguna de las palabras clave (ammo, fight, combat...), y esas palabras son muy frecuentes en el juego.

C: devuelve 5 porque le pedimos top-5 (KNN puede devolver más si aumentamos k). Lo relevante es que genera un ranking continuo por similitud, no un filtro binario.

**3) ¿Qué tal es el orden que devuelve?**

A: el orden no refleja grado de similitud. Las 5 cartas devueltas son todas relevantes (armas de fuego guardian), pero no hay criterio para decir cuál se parece más a Shotgun. Springfield M1903 sale primera simplemente por el orden interno de Redis, no porque sea la más similar.

B: el orden viene dado por la relevancia textual de RediSearch (TF-IDF). Salen en orden según cómo coincidan léxicamente con Shotgun (ammo, fight, combat, damage, attack). El orden es razonable pero prioriza coincidencia de palabras, no similitud funcional.

C: es el mejor ordenado. El score de distancia coseno refleja directamente cuánto se parece cada carta. Old Shotgun (0.1506) es claramente la más cercana, seguida de Sawed-Off Shotgun (0.2253), y luego rifles similares. El orden tiene sentido intuitivo y cuantitativo.



**Nuestra opinión justificada sobre los resultados obtenidos:**

Con los resultados obtenidos se aprecia claramente qué aporta cada enfoque.

En A (metadatos), el recomendador se apoya en información estructurada (facción y traits). Esto produce recomendaciones muy justificables: todas las cartas devueltas son armas de fuego de la facción guardian, que es exactamente lo que comparten con Shotgun. Sin embargo, el conjunto es limitado y el orden no aporta información sobre el grado de similitud. Además, faltan características importantes como el texto de la carta, que es lo que realmente define su funcionalidad. 

En B (full-text), el criterio cambia radicalmente: aquí la similitud viene determinada por el vocabulario del texto. Esto genera un conjunto enorme de candidatos (más de 4000) porque palabras como "fight", "combat" o "damage" aparecen en casi la mitad de las cartas del juego. Los primeros resultados son razonables, pero el método no distingue bien entre una carta que realmente funciona como una escopeta y otra que simplemente menciona las mismas palabras. También se observa que al eliminar las stopwords con nltk y buscar solo en el texto (sin el nombre), los resultados mejoran respecto a incluir el nombre, que suele ser demasiado específico.

Por último, C (búsqueda semántica) es el enfoque que mejor captura la idea de "cartas parecidas" tal como la entendería un jugador. Los 5 resultados son todos armas de fuego con mecánicas prácticamente idénticas a Shotgun (gastar munición, atacar, daño variable), y el orden del ranking es coherente: Old Shotgun (score 0.15) es la más cercana, seguida de Sawed-Off Shotgun (0.23), y luego rifles con mecánicas progresivamente menos similares. Al trabajar con embeddings, la comparación se hace a nivel de significado, no de palabras exactas ni de etiquetas.

En resumen, A es la opción más simple e interpretable, pero limitada en alcance y sin orden significativo. B amplía mucho el abanico pero con demasiado ruido y un orden condicionado por frecuencia léxica. C ofrece las recomendaciones más precisas y el mejor ranking, a costa de necesitar un modelo de embeddings.